# معمل التجارب: أساسيات التقطيع الدلالي المتقدم (المحاضرة 37)
## القسم الأول: الشرح النظري والمعماري

في هذا المعمل العملي:
1. نفحص عملياً كيف يتسبب التقطيع الحجمي الثابت (`RecursiveCharacterTextSplitter`) في تشويه السياق وخلط الموضوعات.
2. نطبق المراحل الأربع لخوارزمية التقطيع الدلالي (`Semantic Chunking`): تجزئة الجمل، حساب المتجهات، قياس التشابه الزوجي، والتقطيع عند العتبة.


## القسم الثاني: الكود المرجعي للتطبيق والتجربة
شغل الخلايا التالية لمتابعة مقارنة التقطيع الحجمي مع التقطيع الدلالي، ثم طبق بنفسك في مساحة التجربة الحرة.

In [1]:
# 1. استدعاء المكتبات وإعداد النص
import re
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

sample_text = (
    "LangChain is a framework to build RAG applications. "
    "LangChain can also be used to build Generative AI applications and multi-agent workflows. "
    "Paris is the capital of France. "
    "The Eiffel Tower is located in Paris, France and is a popular destination."
)

print(f"[INFO] Corpus length: {len(sample_text)} characters.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Corpus length: 248 characters.


In [2]:
# 2. تجربة التقطيع الحجمي وملاحظة المشكلة
fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=115, chunk_overlap=15)
fixed_chunks = fixed_splitter.split_text(sample_text)

print(f"[FIXED-SIZE] Produced {len(fixed_chunks)} chunks:")
for i, c in enumerate(fixed_chunks, 1):
    print(f"[{i}] '{c}'")

[FIXED-SIZE] Produced 3 chunks:
[1] 'LangChain is a framework to build RAG applications. LangChain can also be used to build Generative AI applications'
[2] 'applications and multi-agent workflows. Paris is the capital of France. The Eiffel Tower is located in Paris,'
[3] 'in Paris, France and is a popular destination.'


In [3]:
# 3. التقطيع الدلالي النقي
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", sample_text) if s.strip()]
vectors = np.array(embeddings_model.embed_documents(sentences))

def cosine_sim(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

similarities = [cosine_sim(vectors[i], vectors[i + 1]) for i in range(len(vectors) - 1)]

threshold = 0.35
semantic_chunks = []
current_chunk = [sentences[0]]

for i, sim in enumerate(similarities):
    if sim >= threshold:
        current_chunk.append(sentences[i + 1])
    else:
        semantic_chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i + 1]]
if current_chunk:
    semantic_chunks.append(" ".join(current_chunk))

print(f"\n[SEMANTIC CHUNKS] Produced {len(semantic_chunks)} pure chunks:")
for idx, c in enumerate(semantic_chunks, 1):
    print(f"[{idx}] {c}\n")


[SEMANTIC CHUNKS] Produced 2 pure chunks:
[1] LangChain is a framework to build RAG applications. LangChain can also be used to build Generative AI applications and multi-agent workflows.

[2] Paris is the capital of France. The Eiffel Tower is located in Paris, France and is a popular destination.



---
## القسم الثالث: مساحة التجربة العملية الخاصة بك
استخدم الخلية التالية لتجربة نصوصك الخاصة أو تعديل العتبة.

In [4]:
# مساحتك للتجربة والتطبيق العملي:
# جرب استدعاء نموذج التضمين وتقطيع نص مخصص من عندك وشغله هنا:

